In [0]:
# Databricks notebook source
# tutorial_name: 01-Day7-Structured-Streaming-Delta
# notebookName: 01-Day7-Structured-Streaming-Delta
# COMMAND ----------

# DBTITLE 1,Paths
notepoint_rel = "hands-on/day-07/notebooks/01-Day7-Structured-Streaming-Delta.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
DAY7_PREFIX = f"{BASE_PATH}day07-{STUDENT_ID}"
STREAM_OUT = f"{DAY7_PREFIX}/stream/rate_to_delta"
STREAM_CP = f"{DAY7_PREFIX}/checkpoints/rate_to_delta"
# COMMAND ----------

# DBTITLE 1,Prerequisite check
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("✓ Day 5 tables found (P_BASIC)")
except Exception as e:
    print(f"✗ Day 5 tables not found: {e}")
    print("Please complete Day 5 notebook 01 first.")

print("STREAM_OUT:", STREAM_OUT)
print("STREAM_CP:", STREAM_CP)
# COMMAND ----------


## Day 7 — Item 19: Structured Streaming with Delta

Syllabus item 19 — streaming read, streaming write, checkpointing, Change Data Feed; hands-on **streaming ingestion pipeline**.

Focus: **Streaming read** (`readStream` on the built-in **rate** source), **streaming write** to Delta (`writeStream`, `append`), **checkpoint** under `day07-{STUDENT_ID}`, multiple commits (including stop/restart-style runs with the same checkpoint), **CDF** (`ALTER TABLE` + `readChangeFeed` / optional `table_changes`). The **rate** source keeps shared classrooms predictable; for **Auto Loader** / file landing zones, use `labs.md` (optional snippet).

Run cells **top to bottom** on a fresh `STREAM_OUT` or after cleaning `DAY7_PREFIX` if you repeat the full demo.


### Lab 1 — Micro-batch stream: rate → Delta

`availableNow=True` processes available data then stops (good for class demos). For a long-running job, use `.trigger(processingTime="10 seconds")` instead.

**Checkpointing (course topic):** `checkpointLocation` stores progress so a **restarted** query can continue without reprocessing everything (exactly-once semantics with idempotent sinks). Later labs reuse the same checkpoint path across more micro-batches—observe **version** growth in `DESCRIBE HISTORY`.


In [0]:
from pyspark.sql.functions import col

rate_df = (
    spark.readStream.format("rate")
    .option("rowsPerSecond", 3)
    .option("numPartitions", 1)
    .load()
)
stream_df = rate_df.withColumn("mod100", col("value") % 100)

try:
    for s in spark.streams.active:
        if getattr(s, "name", None) == "rate_to_delta":
            s.stop()
except Exception:
    pass

q = (
    stream_df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", STREAM_CP)
    .queryName("rate_to_delta")
    .trigger(availableNow=True)
    .start(STREAM_OUT)
)
q.awaitTermination()
print("Wrote micro-batches to:", STREAM_OUT)


### Lab 2 — Read the sink in batch mode

Then inspect **version history** after the first write (version **0** = first commit; CDF is not on yet).


In [0]:
batch = spark.read.format("delta").load(STREAM_OUT)
print("row count:", batch.count())
batch.orderBy("timestamp").display(5, truncate=False)

print("--- DESCRIBE HISTORY (after Lab 1) ---")
spark.sql(f"DESCRIBE HISTORY delta.`{STREAM_OUT}`").select(
    "version", "operation", "operationMetrics"
).display(20, truncate=80)


### Lab 3 — Enable Change Data Feed (CDF)

After CDF is on, **new** commits record changes. Versions written **before** CDF was turned on (usually version **0**) have no change feed. When you read the feed, set `startingVersion` to **1 or higher**—not **0**—or you will see `DELTA_MISSING_CHANGE_DATA`.


In [0]:
spark.sql(
    f"ALTER TABLE delta.`{STREAM_OUT}` SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')"
)
print("CDF enabled on sink table.")


### Lab 4 — More table versions (after CDF is on)

Each step creates a **new Delta version**. Together they give `insert`, `delete`, and `update` rows in the change feed (Lab 5).

1. Two more **streaming** `availableNow` appends (same checkpoint).
2. **Batch append**: re-read a few rows and append (duplicates are fine for the demo).
3. **DELETE** one row (minimum `value`).
4. **UPDATE** rows with `mod100 = 0` to `mod100 = 99` (Delta Python API).


In [0]:
from delta.tables import DeltaTable


def _stop_rate_query():
    try:
        for s in spark.streams.active:
            if getattr(s, "name", None) == "rate_to_delta":
                s.stop()
    except Exception:
        pass


def _run_stream_append(label: str):
    _stop_rate_query()
    q = (
        stream_df.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", STREAM_CP)
        .queryName("rate_to_delta")
        .trigger(availableNow=True)
        .start(STREAM_OUT)
    )
    q.awaitTermination()
    print(label)


_run_stream_append("Version: streaming micro-batch #2 (after CDF enabled)")
_run_stream_append("Version: streaming micro-batch #3")

dup = spark.read.format("delta").load(STREAM_OUT).limit(5)
dup.write.format("delta").mode("append").save(STREAM_OUT)
print("Version: batch append (+5 duplicate rows)")

vmin_row = (
    spark.read.format("delta")
    .load(STREAM_OUT)
    .selectExpr("min(value) as vmin")
    .collect()[0]
)
vmin = vmin_row["vmin"]
if vmin is not None:
    spark.sql(f"DELETE FROM delta.`{STREAM_OUT}` WHERE value = {int(vmin)}")
    print("Version: DELETE one row (former min value)")
else:
    print("Skip DELETE: empty table")

dt = DeltaTable.forPath(spark, STREAM_OUT)
dt.update(condition="mod100 = 0", set={"mod100": "99"})
print("Version: UPDATE (mod100 = 0 -> 99 for matching rows)")

print("--- DESCRIBE HISTORY (after Lab 4) ---")
spark.sql(f"DESCRIBE HISTORY delta.`{STREAM_OUT}`").select(
    "version", "operation", "operationMetrics"
).display(30, truncate=80)


### Lab 5 — Read the change feed

Version **0** was written **before** CDF existed, so `startingVersion=0` raises `DELTA_MISSING_CHANGE_DATA`. The cell below picks the **smallest** `startingVersion` that works (usually **1** or **2**).

You should see **`insert`**, **`delete`**, and **`update`** `_change_type` values from Lab 4.


In [0]:
hist = spark.sql(f"DESCRIBE HISTORY delta.`{STREAM_OUT}`").collect()
max_v = max(r["version"] for r in hist)

cdf_start = None
last_err = None
for v in range(1, max_v + 1):
    try:
        _test = (
            spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", v)
            .load(STREAM_OUT)
        )
        _test.limit(1).count()
        cdf_start = v
        break
    except Exception as e:
        last_err = e
        continue

if cdf_start is None:
    raise last_err if last_err is not None else RuntimeError("Could not read change feed; check CDF is enabled and DESCRIBE HISTORY.")

print(f"readChangeFeed startingVersion={cdf_start} (table max version={max_v})")

cdf_df = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", cdf_start)
    .load(STREAM_OUT)
)
cdf_df.orderBy("_commit_version", "_commit_timestamp", "_change_type").display(50, truncate=False)

print("--- Same data via SQL table_changes (if supported on your DBR) ---")
try:
    spark.sql(
        f"SELECT * FROM table_changes('delta.`{STREAM_OUT}`', {cdf_start})"
    ).orderBy("_commit_version", "_commit_timestamp").display(50, truncate=False)
except Exception as e:
    print("table_changes skipped:", e)


### Optional — Clean up (your prefix only)

Uncomment to remove checkpoint and stream output under your Day 7 prefix when you are done experimenting.


In [0]:
# dbutils.fs.rm(DAY7_PREFIX, recurse=True)
